# 📚 Конспект: Python File I/O, Context Managers та JSON

> Урок 11 — конспект: робота з файлами, безпечне управління ресурсами,
> серіалізація даних через JSON та форматування виводу.

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Розминка | RAM vs Disk, персистентність |
| 2 | Основи файлів | File Handle, OS, `open()` |
| 3 | Режими `open()` | `r`, `w`, `a`, `encoding` |
| 4 | Читання файлів | File Pointer, `.tell()`, `.seek()`, ітерація |
| 5 | Запис файлів | `.write()`, `\n`, типи |
| 6 | Context Managers | `with open(...) as f:`, Resource Leak |
| 7 | JSON | Серіалізація, десеріалізація, `json.dump/load` |
| 7б | Типи даних JSON | String, Number, Boolean, Null, Array, Object — нюанси |
| 7в | JSON у реальному світі | REST API, vs XML, AJAX, SPA |
| 8 | Форматування виводу | f-strings, `:<`, `:.2f` |
| 9 | Практичний проєкт | Телефонна книга (Phonebook) |
| 10 | Задачі з дебагом | 3 типові помилки |
| 11 | Рефлексія | Підсумки, питання, реальні use cases |

---

## 🧠 Ментальна модель: Файл як зовнішній стан

```
Python Program (RAM — тимчасово)
    └── змінні, списки, словники → зникають при завершенні програми

Hard Drive / SSD (Disk — постійно)
    └── файли → залишаються після завершення програми

OS (Operating System) — охоронець диску
    └── Python не може торкатись диску напряму
    └── Python просить OS: "відкрий з'єднання до файлу"
    └── OS повертає: File Handle (= посилання на відкрите з'єднання)
```

> **Ключова ідея:** Змінні живуть у RAM. RAM — тимчасова та volatile.
> Коли програма завершується, OS звільняє пам'ять, і дані зникають.
> Щоб зберегти дані назавжди — треба записати їх на Disk (файл).

---

## 📂 Частина 1: Розминка — RAM vs Disk

**Питання для роздумів:** Уяви словник Python де зберігаються рекорди гри:
`scores = {"Alice": 150, "Bob": 120}`. Коли програма завершиться і термінал
закриється — що станеться з `scores`? Куди він зникне?

**Відповідь:**
- Змінні живуть у **RAM (Random Access Memory)** — оперативна пам'ять
- RAM — тимчасова і volatile (енергозалежна)
- При завершенні програми OS звільняє ту ділянку пам'яті → дані **зникають назавжди**
- Щоб зберегти дані постійно → треба записати їх на **Disk (HDD/SSD)**
- Файл — це іменований шматок простору на диску, яким керує OS

Відеоігри пам'ятають твій прогрес, бо **записують save-файл на диск**!

---

## 📂 Частина 2: Основи файлів — `open()` і File Handle

**Передбачення:** що станеться, якщо ми спробуємо відкрити файл,
якого не існує, у режимі `r` (читання)?

In [ ]:
# Спроба відкрити неіснуючий файл у режимі читання
# OS відхиляє запит → Python піднімає FileNotFoundError
try:
    file_handle = open("ghost_data.txt", "r")
    print("Файл відкрито успішно!")
except FileNotFoundError as e:
    print(f"❌ FileNotFoundError: {e}")
    print("Не можна читати те, чого не існує!")

---

## 📂 Частина 3: `open()` та режими відкриття

### Синтаксис

```python
open(filename, mode, encoding)
```

### Основні режими

| Режим | Призначення | Якщо файл є | Якщо файлу немає |
|-------|-------------|-------------|------------------|
| `r` | Тільки читання (default) | Відкриває | `FileNotFoundError` |
| `w` | Тільки запис | **⚠️ СТИРАЄ** вміст! | Створює новий |
| `a` | Додавання в кінець | Дописує в кінець | Створює новий |
| `r+` | Читання і запис | Відкриває | `FileNotFoundError` |

### Важливо: encoding

```python
open("file.txt", "w", encoding="utf-8")  # ← завжди вказуй!
```

UTF-8 каже Python як перетворити бінарні 0 і 1 диску на читабельний текст.
Потрібно для emoji та не-англійських символів (кирилиця!).

In [ ]:
# ДЕМОНСТРАЦІЯ: режим 'w' є деструктивним!
# Передбачення: що буде в test.txt після цього коду?

f = open("test.txt", "w", encoding="utf-8")
f.write("Перший рядок.")
f.close()

# Відкриваємо ЗНОВУ у режимі 'w' — це повністю стирає попередній вміст!
f = open("test.txt", "w", encoding="utf-8")
f.write("Другий рядок.")
f.close()

# Перевіримо вміст
f = open("test.txt", "r", encoding="utf-8")
content = f.read()
f.close()

print(f"Вміст файлу: {content!r}")
print("⚠️ 'Перший рядок.' зникнув! Режим 'w' стирає все!")

In [ ]:
# Задача: відкрити my_diary.txt у 'w', записати "Day 1: ...",
# потім дописати у режимі 'a' "Day 2: ..."

# Крок 1: запис (w)
f = open("my_diary.txt", "w", encoding="utf-8")
f.write("Day 1: I learned about files.")
f.close()

# Крок 2: дописування (a) — не стирає!
f = open("my_diary.txt", "a", encoding="utf-8")
f.write(" Day 2: I survived.")
f.close()

# Перевірка
f = open("my_diary.txt", "r", encoding="utf-8")
print(f.read())
f.close()

---

## 📂 Частина 4: Читання файлів та File Pointer

**Передбачення:** якщо ми двічі поспіль викликаємо `.read()` — що поверне другий виклик?

In [ ]:
# Демонстрація File Pointer (курсору файлу)
f = open("test.txt", "r", encoding="utf-8")

print("--- Перше читання ---")
print(f.read())   # курсор переміщається в кінець

print("--- Друге читання ---")
print(repr(f.read()))  # порожній рядок! Курсор вже в кінці.

f.close()

print()
print("Чому? Курсор (file pointer) після .read() стоїть В КІНЦІ файлу.")
print("Нема що читати → повертає порожній рядок!")

In [ ]:
# .tell() та .seek() — управління курсором
f = open("test.txt", "r", encoding="utf-8")

print(f"Позиція курсору: {f.tell()}")     # 0 — початок

line = f.readline()                         # читаємо один рядок
print(f"Прочитано рядок: {line!r}")
print(f"Курсор тепер на: {f.tell()}")     # переміщений

f.seek(0)                                   # повертаємо на початок
print(f"Після seek(0): {f.tell()}")       # знову 0
print(f"Читаємо знову: {f.read()!r}")

f.close()

In [ ]:
# ✅ Найефективніший спосіб читання великих файлів — ітерація по рядках
# Читає по одному рядку (лінивий) — не завантажує весь файл у RAM!

f = open("test.txt", "r", encoding="utf-8")
for line in f:
    print("РЯДОК:", line.strip())   # .strip() видаляє прихований \n
f.close()

print()
print("⚠️ Якщо файл 10 ГБ і викликати .read() — завантажиш все у RAM → краш!")
print("   Ітерація по рядках — безпечна альтернатива.")

---

## 📂 Частина 5: Запис файлів та типи даних

**Передбачення:** що станеться якщо ми спробуємо записати ціле число `100` у текстовий файл?

In [ ]:
# ❌ Спроба записати ціле число напряму
try:
    f = open("numbers.txt", "w", encoding="utf-8")
    f.write(100)   # TypeError!
    f.close()
except TypeError as e:
    print(f"❌ TypeError: {e}")
    f.close()

print()
print("Текстові файли розуміють ТІЛЬКИ рядки.")
print("Конвертуй через str() перед записом!")

In [ ]:
# ✅ Правильний запис: конвертуємо в str, додаємо \n вручну
# .write() НЕ додає новий рядок автоматично (на відміну від print()!)

names = ["Alice", "Bob", "Charlie"]

f = open("names.txt", "w", encoding="utf-8")
for name in names:
    f.write(name + "\n")   # ← обов'язково \n!
f.close()

# Перевірка
f = open("names.txt", "r", encoding="utf-8")
print(f.read())
f.close()

---

## 🔒 Частина 6: Context Managers — `with open(...) as f:`

**Передбачення:** якщо `10 / 0` викликає помилку і зупиняє програму,
чи виконається `f.close()`? Чому це небезпечно?

In [ ]:
# ❌ Небезпечний код — Resource Leak
try:
    f = open("danger.txt", "w", encoding="utf-8")
    f.write("Починаємо ризикований код...")
    result = 10 / 0   # КРАХ!
    f.close()         # ← ніколи не досягається!
except ZeroDivisionError as e:
    print(f"❌ Помилка: {e}")
    print(f"Файл закритий? {f.closed}")
    f.close()   # треба закривати вручну в except!

print()
print("Resource Leak: OS може тримати файл заблокованим")
print("або дані можуть не записатись на диск!")

In [ ]:
# ✅ Правильний підхід — Context Manager (with)
# 'with' — безпечна кімната. Виходячи з неї (чи нормально, чи через помилку)
# двері ЗАВЖДИ замкнуться (файл ЗАВЖДИ закриється).

with open("safe.txt", "w", encoding="utf-8") as f:
    f.write("Це абсолютно безпечно.\n")
    # f.close() НЕ потрібен — викликається автоматично при виході з блоку!

print(f"Файл закритий автоматично: {f.closed}")
print()
print("Правило: ЗАВЖДИ використовуй 'with open(...) as f:'")
print("Ніколи не пиши просто open() без with!")

In [ ]:
# Задача: переписати код з names.txt за допомогою 'with'

names = ["Alice", "Bob", "Charlie"]

with open("names.txt", "w", encoding="utf-8") as f:
    for name in names:
        f.write(name + "\n")

# Читання теж через with
with open("names.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(content)

---

## 📄 Частина 7: JSON — Серіалізація та десеріалізація

**Проблема:** якщо зберегти словник як `str(my_dict)`, Python забуде що це словник.

**Рішення: JSON (JavaScript Object Notation)**

JSON — це універсальний текстовий формат обміну даними.

### Відповідність типів Python ↔ JSON

| Python | JSON |
|--------|------|
| `dict` | `{}` object |
| `list`, `tuple` | `[]` array |
| `str` | `"string"` |
| `int`, `float` | `number` |
| `True` / `False` | `true` / `false` |
| `None` | `null` |
| `datetime` | ❌ немає — треба `.isoformat()` |

### Ключові функції

```python
import json

# Серіалізація (Python → JSON)
json.dumps(obj)          # Python об'єкт → JSON рядок (s = string)
json.dump(obj, file)     # Python об'єкт → JSON файл

# Десеріалізація (JSON → Python)
json.loads(string)       # JSON рядок → Python об'єкт (s = string)
json.load(file)          # JSON файл → Python об'єкт
```

> **Мнемоніка:** `dump`/`dumps` — **D**ump (скидаємо дані). `load`/`loads` — **L**oad (завантажуємо).
> Суфікс `s` = `string` (рядок, а не файл).

In [ ]:
import json

# Проблема: str() втрачає тип
my_data = {"name": "Alice", "score": 150}
with open("data.txt", "w", encoding="utf-8") as f:
    f.write(str(my_data))   # ← поганий підхід

with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()
    print(f"Тип: {type(text)}  ← це рядок, не словник!")
    print(f"Вміст: {text}")

In [ ]:
import json

# ✅ Правильний підхід: json.dump / json.load

# Збереження на диск (серіалізація)
my_data = {"name": "Alice", "score": 150}
with open("save_game.json", "w", encoding="utf-8") as f:
    json.dump(my_data, f, indent=4)   # indent=4 робить файл красивим!

# Завантаження з диску (десеріалізація)
with open("save_game.json", "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

print(f"Тип: {type(loaded_data)}  ← це знову dict!")  # <class 'dict'>
print(f"name: {loaded_data['name']}")
print(f"score: {loaded_data['score']}")

# Переглянемо вміст файлу
with open("save_game.json", "r", encoding="utf-8") as f:
    print(f"\nВміст JSON файлу:\n{f.read()}")

In [ ]:
import json

# ⚠️ ОБМЕЖЕННЯ: ключі JSON ПОВИНІ бути рядками!
# Tuple як ключ словника → json.dump впаде з TypeError

bad_data = {("Alice", "Smith"): 150}
try:
    with open("crash.json", "w") as f:
        json.dump(bad_data, f)
except TypeError as e:
    print(f"❌ TypeError: {e}")

# ✅ Рішення: перетворити ключ у рядок
first_name, last_name = "Alice", "Smith"
good_data = {f"{first_name} {last_name}": 150}
print(f"Правильно: {good_data}")

# Серіалізація успішна!
serialized = json.dumps(good_data)
print(f"JSON: {serialized}")

---

## 📦 Частина 7б: Типи даних JSON — детальний розбір

JSON підтримує лише **6 типів даних**. Це навмисне обмеження — саме воно робить JSON
таким простим і сумісним із будь-якою мовою програмування.

```
JSON підтримує:
  1. String   (рядок)            "hello"
  2. Number   (число)            42  або  3.14
  3. Boolean  (логічне)          true  /  false
  4. Null     (відсутність)      null
  5. Array    (масив)            [...]
  6. Object   (об'єкт)          {...}

JSON НЕ підтримує:
  ❌ datetime   (треба перетворити у рядок: "2026-03-10T12:30:00")
  ❌ tuple      (перетвориться у масив [])
  ❌ set        (немає аналогу)
  ❌ bytes      (треба Base64-кодування)
```

Подивимось на кожен тип уважно.

---

### 1. Рядки (Strings) — найуніверсальніший тип

**Головне правило:** у JSON рядки завжди в **подвійних лапках** `""`.
Одинарні лапки `''` — заборонені. Це одна з найчастіших причин `JSONDecodeError`!

```
Python:  'hello'   або   "hello"    ← обидва варіанти ОК
JSON:    "hello"                     ← ТІЛЬКИ подвійні лапки
```

**Кодування UTF-8** — JSON підтримує будь-які символи: кирилицю, emoji, ієрогліфи.
Але є нюанс: параметр `ensure_ascii=True` (за замовчуванням!) ескейпує не-ASCII символи:

```python
json.dumps({"місто": "Київ"})               # → {"\\u043c\\u0456..." ...}  (ескейп)
json.dumps({"місто": "Київ"}, ensure_ascii=False)  # → {"місто": "Київ"}   (читабельно)
```

**Секрет:** рядки у JSON використовують не тільки для тексту!
У реальних API рядками передають:
- Великі числа, які JavaScript може округлити: `"id": "9999999999999999999"`
- Дати: `"created_at": "2026-03-10T12:30:00Z"`
- Бінарні дані у Base64: `"avatar": "iVBORw0KGgo..."`

In [ ]:
import json

# Демонстрація рядків у JSON

data = {
    "name":       "Київ",                        # кирилиця
    "emoji":      "🍕",                          # emoji
    "created_at": "2026-03-10T12:30:00",         # дата як рядок
    "user_id":    "9007199254740993",             # велике число — безпечніше як рядок!
}

# ensure_ascii=True (default) — ескейпує кирилицю та emoji
print("ensure_ascii=True (default):")
print(json.dumps(data, indent=2))

print()
# ensure_ascii=False — зберігає UTF-8 символи
print("ensure_ascii=False (рекомендовано для читабельності):")
print(json.dumps(data, indent=2, ensure_ascii=False))

---

### 2. Числа (Numbers) — і чому великі числа небезпечні

JSON підтримує і цілі, і дробові числа. Але є підводний камінь.

**Проблема:** JSON не вказує скільки бітів займає число.
Різні мови можуть читати JSON по-різному:

```
Python:      int необмеженої точності → все ОК
JavaScript:  Number = 64-бітний float → максимум безпечне ціле: 9007199254740991
                                        (2^53 - 1 = приблизно 9 квадрильйонів)
```

Якщо передати через JSON число `9007199254740993` у JavaScript → воно може
**округлитись** до `9007199254740992`. Тихо. Без помилки. Це дуже небезпечно для ID!

In [ ]:
import json

# Ціле число — все добре
safe_int = {"count": 42, "price": 24.98}
print("Ціле і дробове:", json.dumps(safe_int))

# ⚠️ Велике число — потенційна проблема для JavaScript
big_id = 9007199254740993   # більше ніж 2^53

data_number = {"transaction_id": big_id}
data_string = {"transaction_id": str(big_id)}   # безпечно як рядок

print()
print("ID як число:", json.dumps(data_number))
print("ID як рядок:", json.dumps(data_string))

print()
print("Яке рішення краще для API?")
print("  Якщо ID може бути дуже великим → передавай як рядок!")
print("  JavaScript прочитає рядок точно, без округлення.")
print()
print(f"  Максимально безпечне ціле для JS: {2**53 - 1:,}")
print(f"  Наш big_id:                        {big_id:,}")
print(f"  Перевищення:                       {big_id > 2**53 - 1}")

---

### 3. Логічні значення (Booleans) — маленька, але важлива деталь

```
Python:   True    False
JSON:     true    false   ← ТІЛЬКИ маленькі літери!
```

`json.dumps()` автоматично конвертує:
- Python `True` → JSON `true`
- Python `False` → JSON `false`

Але якщо ти **вручну** пишеш JSON-рядок — це класична помилка:
```python
json.loads('{"flag": True}')   # JSONDecodeError! True з великої → не JSON
json.loads('{"flag": true}')   # OK
```

### 4. Відсутнє значення (Null) — три різних смисли

Це найтонший момент у JSON. У API `null`, `""` і **відсутність поля** — це три різні речі!

```
{"nickname": null}     → поле є, але значення свідомо ВІДСУТНЄ
{"nickname": ""}      → поле є, значення — ПОРОЖНІЙ РЯДОК
{}                      → поля взагалі НЕМАЄ
```

Наприклад, у профілі користувача:
- Якщо користувач не встановив нікнейм → поле відсутнє
- Якщо користувач видалив нікнейм → `"nickname": null`
- Якщо нікнейм дозволяється порожнім → `"nickname": ""`

Різні API роблять це по-різному — завжди читай документацію!

In [ ]:
import json

# Booleans: Python ↔ JSON
data = {
    "is_active":     True,
    "email_verified": False,
    "premium":       None,   # None → null
}

serialized = json.dumps(data)
print("Python → JSON:")
print(f"  Python: {data}")
print(f"  JSON:   {serialized}")
print()

restored = json.loads(serialized)
print("JSON → Python:")
print(f"  JSON:   {serialized}")
print(f"  Python: {restored}")
print()

# Три різних стани null
null_demo = {
    "nickname_deleted": None,    # свідомо видалено → null
    "nickname_empty":   "",      # порожній рядок
    # "nickname_missing" → поле відсутнє взагалі
}

print("Три різних стани поля 'nickname':")
print(json.dumps(null_demo, indent=2))
print()
print("Перевірка при читанні:")
print(f"  'nickname_deleted' in json: {'nickname_deleted' in null_demo}")
print(f"  'nickname_missing' in json: {'nickname_missing' in null_demo}")
print(f"  null_demo['nickname_deleted'] is None: {null_demo['nickname_deleted'] is None}")

---

### 5. Масиви (Arrays) → Python `list`

Масив у JSON — це впорядкована колекція значень у `[...]`.
Python отримує їх як звичайний `list`.

**Важлива особливість:** JSON-масив може містити значення **різних типів**:
```json
["яблуко", 3.14, true, null, {"key": "val"}]
```

Але в реальній практиці це погана ідея. Якщо в масиві мішанина типів —
код стає складнішим, а помилки важче знайти.

**Правило хорошого тону:** тримай масиви однорідними.
- `[1, 2, 3]` — ✅ усі числа
- `["Kyiv", "Lviv", "Odesa"]` — ✅ усі рядки
- `[1, "два", null, true]` — ⚠️ мішанина → важко читати і обробляти

In [ ]:
import json

# Масиви — однорідні і змішані
uniform_array = ["Kyiv", "Lviv", "Odesa"]
number_array  = [150, 280, 320, 190]
mixed_array   = ["яблуко", 3.14, True, None]   # погана практика

data = {
    "cities":        uniform_array,
    "prices":        number_array,
    "mixed_example": mixed_array,    # показуємо що можна, але не треба
}

print(json.dumps(data, ensure_ascii=False, indent=2))

print()
# Масиви об'єктів — найчастіший реальний сценарій
orders = [
    {"id": 1, "dish": "Pizza", "price": 320},
    {"id": 2, "dish": "Burger", "price": 210},
]
print("Масив об'єктів (типовий API response):")
print(json.dumps(orders, indent=2))

---

### 6. Об'єкти (Objects) → Python `dict`

Об'єкт — це `{"ключ": значення}`. Python отримує як `dict`.

**Головне обмеження: ключі — ТІЛЬКИ рядки.**

```python
# ❌ Ці ключі не можна серіалізувати у JSON:
{("Alice", "Smith"): 150}    # tuple → TypeError
{1: "one", 2: "two"}          # int-ключі → json конвертує у рядки!
```

Цікавий момент з числовими ключами:

In [ ]:
import json

# ⚠️ Числові ключі: json автоматично конвертує у рядки
int_keys = {1: "one", 2: "two", 3: "three"}

serialized = json.dumps(int_keys)
print(f"dict з int-ключами:  {int_keys}")
print(f"JSON:                {serialized}")
print()

restored = json.loads(serialized)
print(f"Після json.loads():  {restored}")
print(f"Ключі тепер рядки:  {list(restored.keys())}")
print(f"int_keys == restored: {int_keys == restored}  ← РІЗНІ! Ключі змінились!")
print()

# ❌ Tuple-ключі → TypeError
try:
    json.dumps({("Alice", "Smith"): 150})
except TypeError as e:
    print(f"TypeError для tuple-ключа: {e}")
print()

# ✅ Правильно — конвертуємо ключ у рядок заздалегідь
good = {"Alice Smith": 150}   # або f"{first} {last}"
print(f"Правильно: {json.dumps(good)}")
print()
print("Правило: якщо ключ не рядок — перетвори в рядок ДО json.dumps()")

---

### 7. Що JSON не підтримує — і як обходити

```python
from datetime import datetime
import json

# ❌ datetime → TypeError
data = {"created_at": datetime.now()}
# json.dumps(data)  →  TypeError: Object of type datetime is not JSON serializable
```

**Рішення для типів, які JSON не підтримує:**

| Тип Python | Проблема | Рішення |
|-----------|----------|--------|
| `datetime` | JSON не знає | `dt.isoformat()` → рядок `"2026-03-10T12:30:00"` |
| `tuple` | → JSON масив (тип губиться) | Прийняти це або перетворити у список |
| `set` | JSON не знає | `list(my_set)` |
| `Decimal` | Округлення | `str(value)` або кастомний encoder |
| `bytes` | JSON не знає | Base64 кодування |

In [ ]:
import json
from datetime import datetime

# ❌ datetime напряму — TypeError
try:
    json.dumps({"time": datetime.now()})
except TypeError as e:
    print(f"❌ TypeError: {e}")

print()

# ✅ Рішення 1: .isoformat() — стандарт ISO 8601
now = datetime.now()
data = {
    "event":      "Замовлення оформлено",
    "created_at": now.isoformat(),            # "2026-03-10T12:30:55.123456"
    "date_only":  now.date().isoformat(),     # "2026-03-10"
}
print("Дата як рядок ISO 8601:")
print(json.dumps(data, ensure_ascii=False, indent=2))

print()

# ✅ Рішення 2: custom default функція для json.dumps
def json_default(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(list(obj))   # set → список
    raise TypeError(f"Тип {type(obj).__name__} не серіалізується")

messy_data = {
    "updated_at": datetime(2026, 3, 10, 12, 30),
    "tags":       {"python", "beginner", "file-io"},   # set!
}
print("Custom default (datetime + set):")
print(json.dumps(messy_data, default=json_default, ensure_ascii=False, indent=2))

---

## 🌐 Частина 7в: JSON у реальному світі — як він став стандартом

### Чому JSON переміг XML

До JSON інтернет-системи спілкувались через **XML**. Ось порівняння:

```xml
<!-- Один і той самий об'єкт у XML -->
<user>
  <name>Alice</name>
  <age>28</age>
  <active>true</active>
</user>
```

```json
{"name": "Alice", "age": 28, "active": true}
```

XML займав **на 36% більше символів** для тих самих даних.
Кожне поле — два теги (відкриваючий і закриваючий). Для мобільних пристроїв
і повільних мереж це реально мало значення.

JSON виграв через три речі:
- **Компактність** — менший розмір = швидша передача
- **Простота** — немає тегів, атрибутів, просторів імен
- **Натуральність для JavaScript** — `JSON.parse()` вбудований у будь-який браузер

### Як JSON і REST API змінили веб

**REST API** (Representational State Transfer) — це архітектурний стиль,
де клієнт і сервер обмінюються даними через HTTP, і JSON став форматом за замовчуванням.

Ось що відбувається коли ти відкриваєш додаток:

```
Твій телефон (React Native)              Сервер (Python / Node.js)
─────────────────────                    ────────────────────────
GET /api/orders  ──────────────────────► читає БД
                 ◄──────────────────────  повертає JSON
                   {
                     "orders": [...]       ← frontend бачить список замовлень
                   }
```

Саме такий JSON ми і записали у `report.json` — тільки у реальному API
він не зберігається у файл, а **відправляється напряму у відповідь на HTTP-запит**.

### Великі API, які використовують JSON

- **Twitter/X API** → отримуєш список твітів у JSON
- **GitHub API** → дані про репозиторії, коміти у JSON
- **Google Maps API** → координати, адреси у JSON
- **OpenWeatherMap** → погода у JSON
- **Будь-який сучасний веб-застосунок** → JSON між фронтом і беком

### AJAX і Single Page Applications

**AJAX** (Asynchronous JavaScript and XML) — технологія, яка дозволяє
оновлювати частину сторінки без перезавантаження. "X" у назві — XML,
але сьогодні **майже завжди використовується JSON** (жартівливо кажуть AJAJ).

```
Стара веб-розробка:
  Клацнув кнопку → завантажилась нова сторінка повністю

Сучасна (SPA — Single Page Application):
  Клацнув кнопку → JavaScript зробив запит → отримав JSON → оновив тільки потрібний блок
```

Саме тому у **React, Vue, Angular** — frontend говорить з backend виключно через JSON.
Наш `report.json` — це саме те, що реальний React-компонент прочитає і відобразить на сторінці.

---

## 🎨 Частина 8: Форматування виводу — f-strings

### Специфікатори формату

```python
f"{value:<width}"   # вирівнювання по лівому краю, ширина width
f"{value:>width}"   # вирівнювання по правому краю
f"{value:^width}"   # центрування
f"{value:.2f}"      # float з 2 знаками після крапки
f"{value:,}"        # роздільник тисяч (1,000,000)
f"{value:0>5d}"     # ціле число, заповнення нулями, ширина 5
```

In [ ]:
# Форматування колонок — вирівнювання та десяткові знаки

contacts = [
    ("Alice Smith", "050-123-4567", 1500.5),
    ("Bob Jones", "097-987-6543", 250.0),
    ("Charlie Brown", "063-555-1234", 99999.99),
]

# Заголовок таблиці
print(f"| {'Ім\'я':<15} | {'Телефон':<14} | {'Баланс':>12} |")
print("|" + "-"*17 + "|" + "-"*16 + "|" + "-"*14 + "|")

for name, phone, balance in contacts:
    print(f"| {name:<15} | {phone:<14} | ${balance:>10.2f} |")

print()
print("f-string форматування:")
print(f"  ':<15'  — лівий край, 15 символів")
print(f"  ':>10.2f' — правий край, 10 символів, 2 знаки після крапки")

---

## 💼 Частина 9: Практичний проєкт — Телефонна книга

**Мета:** побудувати персистентну телефонну книгу.

**Вимоги:**
1. Ключ словника — рядок `f"{first_name} {last_name}"` (бо JSON вимагає рядкові ключі)
2. Значення — словник `{"phone": ..., "balance": ...}`
3. При старті — спробувати завантажити `phonebook.json`, при помилці — почати з `{}`
4. Меню у `while True`: Додати / Знайти / Вийти
5. При виході — зберегти дані у `phonebook.json`

In [ ]:
import json

PHONEBOOK_FILE = "phonebook.json"

# 1. Завантаження існуючих даних або старт з нуля
try:
    with open(PHONEBOOK_FILE, "r", encoding="utf-8") as f:
        phonebook = json.load(f)
    print("Завантажено існуючу телефонну книгу!")
except FileNotFoundError:
    print("Телефонної книги не знайдено. Починаємо з нуля.")
    phonebook = {}

# 2. Головний цикл
while True:
    print("\n--- МЕНЮ ТЕЛЕФОННОЇ КНИГИ ---")
    print("1. Додати контакт")
    print("2. Знайти контакт")
    print("3. Вийти")

    choice = input("Введіть вибір: ")

    if choice == "1":
        first = input("Ім'я: ")
        last = input("Прізвище: ")
        phone = input("Телефон: ")
        balance = float(input("Баланс рахунку: "))

        key = f"{first} {last}"              # рядковий ключ для JSON
        phonebook[key] = {"phone": phone, "balance": balance}
        print(f"✅ Контакт '{key}' додано!")

    elif choice == "2":
        search_name = input("Введіть повне ім'я: ")
        if search_name in phonebook:
            contact = phonebook[search_name]
            print(f"| {'Ім\'я':<15} | {'Телефон':<14} | {'Баланс':>10} |")
            print(f"| {search_name:<15} | {contact['phone']:<14} | ${contact['balance']:>9.2f} |")
        else:
            print(f"❌ '{search_name}' не знайдено")

    elif choice == "3":
        with open(PHONEBOOK_FILE, "w", encoding="utf-8") as f:
            json.dump(phonebook, f, ensure_ascii=False, indent=4)
        print("💾 Дані збережено. До побачення!")
        break
    else:
        print("Невірний вибір. Спробуй ще раз.")

---

## 🐛 Частина 10: Задачі з дебагу

Знайди та виправ типові помилки початківців при роботі з файлами і JSON.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🐛 Задача 1: Зниклі дані
# Код нібито записує список у файл, але my_list.txt порожній або
# програма падає. Знайди та виправ помилку!
# ─────────────────────────────────────────────────────────────────

# ❌ ЗЛАМАНИЙ КОД:
# data = ["Apples\n", "Bananas\n", "Cherries\n"]
# f = open("my_list.txt", "w")
# for item in data:
#     f.write(item)
# # БАГ: f.close() відсутній! Дані можуть не записатись.

# ✅ ВИПРАВЛЕНИЙ КОД:
data = ["Apples\n", "Bananas\n", "Cherries\n"]

with open("my_list.txt", "w", encoding="utf-8") as f:  # ← 'with' замість ручного close()
    for item in data:
        f.write(item)
# Файл автоматично закритий і даний записані на диск!

with open("my_list.txt", "r", encoding="utf-8") as f:
    print(f.read())
print("✅ Виправлено: використай 'with open(...) as f:'")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🐛 Задача 2: Нескінченне дописування
# Функція має скидати рахунок до 0 щоразу при виклику.
# Але натомість файл росте: 00000
# ─────────────────────────────────────────────────────────────────

# ❌ ЗЛАМАНИЙ КОД:
def reset_score_broken():
    with open("score.txt", "a") as f:   # ← 'a' ДОПИСУЄ!
        f.write("0")

reset_score_broken()
reset_score_broken()

with open("score.txt", "r") as f:
    print(f"Зламаний: {f.read()!r}")   # '00' — дописало!

# ✅ ВИПРАВЛЕНИЙ КОД:
def reset_score():
    with open("score.txt", "w") as f:   # ← 'w' ПЕРЕЗАПИСУЄ!
        f.write("0")

reset_score()
reset_score()

with open("score.txt", "r") as f:
    print(f"Виправлено: {f.read()!r}")  # '0' — скинуто коректно!

print("✅ Виправлено: режим 'w' (write) замість 'a' (append)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🐛 Задача 3: JSON Decode Error
# Код падає з json.decoder.JSONDecodeError при завантаженні.
# Знайди де баг при записі!
# ─────────────────────────────────────────────────────────────────
import json

my_config = {"volume": 80, "fullscreen": True}

# ❌ ЗЛАМАНИЙ запис:
with open("config.json", "w", encoding="utf-8") as f:
    f.write(str(my_config))   # ← БАГ: str() дає {'volume': 80, 'fullscreen': True}
                               # але JSON вимагає {"volume": 80, "fullscreen": true}
                               # одинарні лапки та True → Python-синтаксис, не JSON!

try:
    with open("config.json", "r", encoding="utf-8") as f:
        json.load(f)
except json.JSONDecodeError as e:
    print(f"❌ JSONDecodeError: {e}")

# ✅ ВИПРАВЛЕНИЙ запис:
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(my_config, f, indent=2)   # ← json.dump замість str()!

with open("config.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(f"✅ Виправлено! Завантажено: {loaded}")
print(f"   fullscreen: {loaded['fullscreen']} (тип: {type(loaded['fullscreen']).__name__})")

---

## 🏁 Частина 11: Рефлексія

### 5 ключових висновків

1. **Memory vs Disk:** Змінні живуть у тимчасовому RAM. Файли живуть на постійному Disk. Програма мусить просити OS відкрити з'єднання.

2. **Режими файлів мають значення:** `w` — деструктивний, перезаписує. `a` — безпечно дописує. `r` — читає, падає якщо файл відсутній.

3. **Context Managers (`with`):** ЗАВЖДИ використовуй `with open(...) as f:` — гарантує закриття файлу навіть при краші програми.

4. **Все — рядок:** текстові файли розуміють лише рядки. Не можна передати `int` або `list` напряму у `.write()`.

5. **JSON — універсальний:** використовуй `json.dump()` та `json.load()` для безшовного перетворення структур даних Python (словники, списки) у текстові файли і назад.

---

### 3 концептуальні питання

1. Якщо викликати `.read()` на файлі 50 ГБ — що станеться з RAM? Яка безпечна альтернатива?

2. Чому Python викидає помилку при спробі використати tuple `("Alice", "Smith")` як ключ словника, що зберігається у JSON?

3. Яка різниця між крашем програми ДО `f.close()` і крашем ВСЕРЕДИНІ блоку `with open(...) as f:`?

---

### 3 реальні use cases

- **Web API:** Python-бекенд спілкується з React/JavaScript фронтендом виключно через JSON.
- **Конфіг-файли:** майже всі Python CLI-інструменти зберігають налаштування (dark mode, ім'я користувача) у локальному `.json` файлі.
- **Data Science:** дата-інженери читають масивні `.csv` або `.txt` файли рядок за рядком (лінивий ітератор) для очищення тексту перед завантаженням у базу даних.